# MintPy Time-Series Interpretation Lab

This notebook is a starting point for interpreting a MintPy result directory after running:

```bash
smallbaselineApp.py mintpy/RidgecrestSenDT71.txt
```

or a similar MintPy template for another example dataset such as the San Francisco Bay stack.

## Learning goals

By the end of this lab, you should be able to:

1. Identify the main MintPy output files.
2. Explain the difference between an interferogram and a date-by-date displacement time series.
3. Interpret line-of-sight (LOS) displacement and velocity maps.
4. Extract point time series from selected pixels.
5. Make cross sections across deformation gradients or faults.
6. Calculate the spatial gradient of LOS velocity and use it to highlight creeping faults.
7. Evaluate which parts of the result are reliable using coherence and temporal coherence.

## Important sign convention

MintPy LOS displacement and velocity are usually reported with this convention:

**positive LOS = motion toward the satellite**

Do not automatically interpret positive LOS as uplift. In a single viewing geometry, LOS motion mixes vertical and horizontal displacement.


## 0. Setup

Run this notebook from inside the MintPy result directory, for example:

```text
RidgecrestSenDT71/
├── timeseries.h5
├── velocity.h5
├── temporalCoherence.h5
├── avgSpatialCoh.h5
├── inputs/
│   ├── ifgramStack.h5
│   └── geometryGeo.h5
└── pic/
```

If your notebook is somewhere else, edit `DATA_DIR`.


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

try:
    from scipy.ndimage import map_coordinates
except Exception as e:
    raise ImportError("This notebook needs scipy. Install it with: conda install -c conda-forge scipy") from e

DATA_DIR = Path("./RidgecrestSenDT71/mintpy").resolve()

# Main MintPy files
TS_FILE = DATA_DIR / "timeseries.h5"
VEL_FILE = DATA_DIR / "velocity.h5"
TCOH_FILE = DATA_DIR / "temporalCoherence.h5"
AVG_COH_FILE = DATA_DIR / "avgSpatialCoh.h5"

# Geometry can be in inputs/geometryGeo.h5 or geometryRadar.h5 depending on the stack
GEOM_CANDIDATES = [
    DATA_DIR / "inputs" / "geometryGeo.h5",
    DATA_DIR / "inputs" / "geometryRadar.h5",
    DATA_DIR / "geometryGeo.h5",
    DATA_DIR / "geometryRadar.h5",
]
GEOM_FILE = next((p for p in GEOM_CANDIDATES if p.exists()), None)

print("Working directory:", DATA_DIR)
print("timeseries.h5 exists:", TS_FILE.exists())
print("velocity.h5 exists:", VEL_FILE.exists())
print("geometry file:", GEOM_FILE)


## 1. Inspect the HDF5 files

MintPy products are HDF5 files. Each file contains one or more datasets plus metadata attributes.

Use this section to inspect what is actually in your files.


In [ ]:
def inspect_h5(path):
    """Print datasets and selected attributes in an HDF5 file."""
    path = Path(path)
    print("\n" + "=" * 80)
    print(path)
    print("=" * 80)
    if not path.exists():
        print("File not found.")
        return

    with h5py.File(path, "r") as f:
        print("Datasets/groups:")
        def printer(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name:35s} shape={obj.shape} dtype={obj.dtype}")
            else:
                print(f"  {name:35s} group")
        f.visititems(printer)

        print("\nSelected attributes:")
        for key in [
            "FILE_TYPE", "UNIT", "REF_DATE", "REF_Y", "REF_X", 
            "REF_LAT", "REF_LON", "HEADING", "WAVELENGTH",
            "X_FIRST", "Y_FIRST", "X_STEP", "Y_STEP", "LENGTH", "WIDTH"
        ]:
            if key in f.attrs:
                print(f"  {key:12s}: {f.attrs[key]}")

for p in [TS_FILE, VEL_FILE, TCOH_FILE, AVG_COH_FILE, GEOM_FILE]:
    if p is not None:
        inspect_h5(p)


### Questions

**QUESTION (5pts): What Datasets are present in the timeseries.hs and velocity.h5 files, respectively? What is the reference date?**

**ANSWER:**

**QUESTION (5pts): What are the units of the timeseries and velocity products? Is this geometry geocoded or radar-coded?**

**ANSWER:**


## 2. Helper functions

These functions load MintPy data, convert units, plot maps, and derive simple latitude/longitude grids where possible.


In [ ]:
def _decode(x):
    if isinstance(x, bytes):
        return x.decode()
    return str(x)

def read_dataset(path, dataset_name=None):
    """
    Read a dataset from an HDF5 file.
    If dataset_name is None and there is only one dataset, read that dataset.
    Returns data, attrs.
    """
    path = Path(path)
    with h5py.File(path, "r") as f:
        attrs = {k: _decode(v) if isinstance(v, bytes) else v for k, v in f.attrs.items()}
        if dataset_name is None:
            dsets = []
            def collect(name, obj):
                if isinstance(obj, h5py.Dataset):
                    dsets.append(name)
            f.visititems(collect)
            if len(dsets) == 1:
                dataset_name = dsets[0]
            else:
                raise ValueError(f"Choose a dataset from {dsets}")
        data = f[dataset_name][:]
    return data, attrs

def read_dates(ts_file=TS_FILE):
    """Read acquisition dates from timeseries.h5."""
    with h5py.File(ts_file, "r") as f:
        if "date" in f:
            dates = [_decode(d) for d in f["date"][:]]
        else:
            raise KeyError("Could not find dataset 'date' in timeseries.h5")
    return dates

def read_timeseries_cm(ts_file=TS_FILE, reference_to_first=True):
    """Read timeseries in cm. Optionally re-reference to the first acquisition date."""
    ts, attrs = read_dataset(ts_file, "timeseries")
    ts_cm = ts.astype("float64") * 100.0  # MintPy displacement is commonly in meters
    if reference_to_first:
        ts_cm = ts_cm - ts_cm[0, :, :]
    dates = read_dates(ts_file)
    return ts_cm, dates, attrs

def read_velocity_cm_per_year(vel_file=VEL_FILE):
    """Read velocity in cm/year."""
    vel, attrs = read_dataset(vel_file, "velocity")
    # MintPy velocity is commonly in m/year; convert to cm/year.
    vel_cm_yr = vel.astype("float64") * 100.0
    return vel_cm_yr, attrs

def read_optional_map(path, dataset_name=None):
    """Read an optional HDF5 map. Return None if file is absent."""
    path = Path(path)
    if not path.exists():
        return None, {}
    if dataset_name is None:
        try:
            return read_dataset(path, None)
        except ValueError:
            # Common names
            for name in ["temporalCoherence", "coherence", "avgSpatialCoh", "mask"]:
                try:
                    return read_dataset(path, name)
                except Exception:
                    pass
            raise
    return read_dataset(path, dataset_name)

def get_geo_grids(attrs, shape):
    """
    Build longitude/latitude grids from MintPy geocoded metadata if available.
    Returns lon2d, lat2d or None, None.
    """
    required = ["X_FIRST", "Y_FIRST", "X_STEP", "Y_STEP"]
    if not all(k in attrs for k in required):
        return None, None

    nrows, ncols = shape
    x_first = float(attrs["X_FIRST"])
    y_first = float(attrs["Y_FIRST"])
    x_step = float(attrs["X_STEP"])
    y_step = float(attrs["Y_STEP"])

    xs = x_first + np.arange(ncols) * x_step
    ys = y_first + np.arange(nrows) * y_step
    lon2d, lat2d = np.meshgrid(xs, ys)
    return lon2d, lat2d

def plot_map(data, title="", label="", cmap="RdBu_r", vlim=None, mask=None, figsize=(9, 6)):
    """Simple map plot with optional symmetric limits."""
    arr = np.array(data, dtype="float64")
    if mask is not None:
        arr = np.where(mask, arr, np.nan)

    if vlim is None:
        finite = arr[np.isfinite(arr)]
        if finite.size:
            p = np.nanpercentile(np.abs(finite), 98)
            vlim = (-p, p)
        else:
            vlim = (-1, 1)
    elif np.isscalar(vlim):
        vlim = (-abs(vlim), abs(vlim))

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(arr, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    ax.set_title(title)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(label)
    plt.show()
    return fig, ax

def print_dates(dates):
    for i, d in enumerate(dates):
        print(f"{i:2d}: {d}")


## 3. Load and plot the basic products


In [ ]:
ts_cm, dates, ts_attrs = read_timeseries_cm(TS_FILE, reference_to_first=True)
vel_cm_yr, vel_attrs = read_velocity_cm_per_year(VEL_FILE)

temporal_coh, _ = read_optional_map(TCOH_FILE)
avg_coh, _ = read_optional_map(AVG_COH_FILE)

print("Time-series shape:", ts_cm.shape, "(date, y, x)")
print("Velocity shape:", vel_cm_yr.shape)
print("Dates:")
print_dates(dates)

print("\nReference date in file:", ts_attrs.get("REF_DATE"))
print("Reference pixel y/x:", ts_attrs.get("REF_Y"), ts_attrs.get("REF_X"))
print("Heading:", ts_attrs.get("HEADING"))


In [ ]:
plot_map(vel_cm_yr, title="LOS velocity", label="cm/year", vlim=None)

if temporal_coh is not None:
    plot_map(temporal_coh, title="Temporal coherence", label="unitless", cmap="viridis", vlim=(0, 1))

if avg_coh is not None:
    plot_map(avg_coh, title="Average spatial coherence", label="unitless", cmap="viridis", vlim=(0, 1))


### Questions

**QUESTION (5pts): What is (approximately) the strongest apparent LOS velocity?**

**ANSWER:**


## 4. Plot displacement maps through time

For an earthquake sequence, the main signal may be a step between pre-event and post-event acquisitions.  
For a creeping-fault or strain-accumulation example, the changes may be gradual and better represented by velocity.


In [ ]:
# Plot all dates, referenced to the first acquisition date.
n = len(dates)
ncols = 4
nrows = int(np.ceil(n / ncols))

finite = ts_cm[np.isfinite(ts_cm)]
vmax = np.nanpercentile(np.abs(finite), 98) if finite.size else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for i, date in enumerate(dates):
    im = axes[i].imshow(ts_cm[i], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[i].set_title(date)
    axes[i].set_xlabel("x")
    axes[i].set_ylabel("y")

for ax in axes[n:]:
    ax.axis("off")

cbar = fig.colorbar(im, ax=axes[:n], shrink=0.85)
cbar.set_label("LOS displacement relative to first date, cm")
plt.show()


### Pre-event versus post-event displacement map

For Ridgecrest, edit `PRE_DATES` and `POST_DATES` below.  
For San Francisco or another slow-deformation stack, this section is less useful than the velocity and gradient maps.


In [ ]:
# Edit these for your dataset.
# Ridgecrest example:
PRE_DATES = ["20190610", "20190622", "20190704"]
POST_DATES = ["20190716", "20190728", "20190809", "20190815"]

available = set(dates)
pre_idx = [dates.index(d) for d in PRE_DATES if d in available]
post_idx = [dates.index(d) for d in POST_DATES if d in available]

if pre_idx and post_idx:
    pre_mean = np.nanmean(ts_cm[pre_idx], axis=0)
    post_mean = np.nanmean(ts_cm[post_idx], axis=0)
    step_cm = post_mean - pre_mean

    plot_map(step_cm, title="Mean post-event minus mean pre-event LOS displacement", label="cm", vlim=None)
else:
    print("The selected pre/post dates are not available in this dataset.")
    print("Available dates:")
    print_dates(dates)


### Questions

**QUESTION (5pts): What is (approximately) the strongest apparent LOS displacement? Is this displacement map dominated by a long-term trend, an event, or noise?**

**ANSWER:**

**QUESTION (10pts): Compare your answers for the strongest velocity vs displacement. Why is there such a large discrepancy? If you were interpreting these scientifically, would you want to use the displacement solution or the velocity solution?**

**ANSWER:**


## 5. LOS and flight-direction arrows

This section adds approximate arrows for:

- satellite flight direction
- positive LOS direction, i.e. ground motion toward the satellite

The arrow is the horizontal projection only. The real LOS vector also has an upward component.


In [ ]:
def get_heading(attrs):
    if "HEADING" not in attrs:
        return None
    return float(attrs["HEADING"])

def add_arrow_azimuth(ax, az_deg, x0=0.80, y0=0.15, length=0.10, text="", color="k"):
    """
    Add an arrow in axes-fraction coordinates.
    az_deg is clockwise from north.
    """
    dx = length * np.sin(np.deg2rad(az_deg))
    dy = length * np.cos(np.deg2rad(az_deg))
    ax.annotate(
        "",
        xy=(x0 + dx, y0 + dy),
        xytext=(x0, y0),
        xycoords="axes fraction",
        arrowprops=dict(arrowstyle="->", linewidth=2, color=color),
    )
    ax.text(
        x0 + dx,
        y0 + dy,
        text,
        transform=ax.transAxes,
        ha="left",
        va="center",
        fontsize=10,
        color=color,
    )

def add_los_and_flight_arrows(ax, attrs):
    heading = get_heading(attrs)
    if heading is None:
        print("No HEADING metadata found; cannot add LOS/flight arrows.")
        return

    # For right-looking SAR, the positive LOS horizontal projection is approximately heading - 90 degrees.
    positive_los_az = (heading - 90.0) % 360.0
    add_arrow_azimuth(ax, heading, x0=0.78, y0=0.25, length=0.10, text=" flight", color="0.3")
    add_arrow_azimuth(ax, positive_los_az, x0=0.78, y0=0.12, length=0.10, text=" +LOS\ntoward sat.", color="k")
    print(f"Satellite heading: {heading:.1f} deg")
    print(f"Approx. positive LOS horizontal azimuth: {positive_los_az:.1f} deg")

# Example plot with arrows
fig, ax = plt.subplots(figsize=(9, 6))
p = np.nanpercentile(np.abs(vel_cm_yr[np.isfinite(vel_cm_yr)]), 98)
im = ax.imshow(vel_cm_yr, cmap="RdBu_r", vmin=-p, vmax=p)
ax.set_title("LOS velocity with approximate flight and +LOS arrows")
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
plt.colorbar(im, ax=ax, label="cm/year")
add_los_and_flight_arrows(ax, vel_attrs)
plt.show()


### LOS questions

**QUESTION (5pts): Is positive LOS displacement the same as uplift? Why or why not?**

**ANSWER:**

**QUESTION (10pts): What additional data would be needed to separate east-west, north-south, and vertical displacement?**

**ANSWER:**


## 6. Point time series

You can do this interactively with MintPy:

```bash
tsview.py timeseries.h5
```

Click on the map to update the time-series panel.

The notebook version below lets you select pixels manually and save plots. Use map plots above to identify interesting pixels.


In [ ]:
def plot_point_timeseries(points_yx, ts_cm=ts_cm, dates=dates, title="Point LOS displacement time series"):
    """
    points_yx: dictionary of {label: (y, x)}
    """
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, (y, x) in points_yx.items():
        y = int(y)
        x = int(x)
        if y < 0 or y >= ts_cm.shape[1] or x < 0 or x >= ts_cm.shape[2]:
            print(f"Skipping {label}: outside raster")
            continue
        ax.plot(dates, ts_cm[:, y, x], marker="o", label=f"{label} ({y}, {x})")
    ax.axhline(0, color="0.5", linewidth=1)
    ax.set_title(title)
    ax.set_ylabel("LOS displacement relative to first date, cm")
    ax.set_xlabel("Acquisition date")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

# Edit these pixels after inspecting your map.
points = {
    "far field": (50, 50),
    "Western Deformation Lobe": (500, 650),
    "Eastern Deformation Lobe": (650, 900),
}

plot_point_timeseries(points)


### Point time-series tasks (using tsview.py). No need to save plots, just explore a bit

1. Select points in each of the three main deformation lobes
3. Select one far-field point that appears stable.
4. Select one low-coherence point.
5. Plot all four time series.

### Question

**QUESTION (10pts): At which points does the straight-line velocity model make sense? Why or why not?**

**ANSWER:**


## 7. Cross sections across deformation features

Use this section to draw cross sections across faults or deformation gradients.

For Ridgecrest, choose profiles crossing the main rupture and secondary strands.  

The simplest input is pixel coordinates:

```python
profile = ((y0, x0), (y1, x1))
```

**CODING TASK (10pts): You will need to fill in the "EDIT THESE PROFILE ENDPOINTS" section below to chose the appropriate cross sections.**


In [ ]:
def sample_profile(data, start_yx, end_yx, n=500, order=1):
    """
    Sample a 2D array along a line from start_yx to end_yx.
    Returns distance in pixels, sampled values, y coordinates, x coordinates.
    """
    y0, x0 = start_yx
    y1, x1 = end_yx
    ys = np.linspace(y0, y1, n)
    xs = np.linspace(x0, x1, n)
    vals = map_coordinates(data, [ys, xs], order=order, mode="nearest")
    dist_pix = np.sqrt((ys - ys[0])**2 + (xs - xs[0])**2)
    return dist_pix, vals, ys, xs

def plot_profile_on_map(data, start_yx, end_yx, title="Profile location", label=""):
    fig, ax = plt.subplots(figsize=(9, 6))
    finite = data[np.isfinite(data)]
    p = np.nanpercentile(np.abs(finite), 98) if finite.size else 1
    im = ax.imshow(data, cmap="RdBu_r", vmin=-p, vmax=p)
    y0, x0 = start_yx
    y1, x1 = end_yx
    ax.plot([x0, x1], [y0, y1], "k-", linewidth=2)
    ax.plot(x0, y0, "ko", label="start")
    ax.plot(x1, y1, "ks", label="end")
    ax.set_title(title)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    ax.legend()
    plt.colorbar(im, ax=ax, label=label)
    plt.show()

def plot_cross_section(data, start_yx, end_yx, n=500, ylabel="Value", title="Cross section", coherence=None):
    dist_pix, vals, ys, xs = sample_profile(data, start_yx, end_yx, n=n)

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(dist_pix, vals, "k-", linewidth=1.5)
    ax1.set_xlabel("Distance along profile, pixels")
    ax1.set_ylabel(ylabel)
    ax1.set_title(title)
    ax1.grid(True, alpha=0.3)

    if coherence is not None:
        _, coh_vals, _, _ = sample_profile(coherence, start_yx, end_yx, n=n)
        ax2 = ax1.twinx()
        ax2.plot(dist_pix, coh_vals, color="0.6", linewidth=1)
        ax2.set_ylabel("Coherence / temporal coherence")
        ax2.set_ylim(0, 1)

    plt.show()
    return dist_pix, vals

def estimate_pixel_spacing_km(attrs, shape):
    """
    Estimate pixel spacing in km from MintPy spatial metadata.

    Handles two common cases:

    1. Geographic coordinates:
       X/Y are longitude/latitude in degrees, e.g. X_STEP ~ 0.0008.
       Convert degrees to km.

    2. Projected coordinates:
       X/Y are map coordinates in meters, e.g. UTM with X_STEP = 80, Y_STEP = -80.
       Convert meters to km.

    If metadata are unavailable, return pixel units.
    """
    if not all(k in attrs for k in ["X_STEP", "Y_STEP", "X_FIRST", "Y_FIRST"]):
        return 1.0, 1.0, "pixel"

    x_step = abs(float(attrs["X_STEP"]))
    y_step = abs(float(attrs["Y_STEP"]))
    x_first = float(attrs["X_FIRST"])
    y_first = float(attrs["Y_FIRST"])

    looks_like_lonlat = (
        x_step < 1.0 and y_step < 1.0 and
        -180.0 <= x_first <= 180.0 and
        -90.0 <= y_first <= 90.0
    )

    if looks_like_lonlat:
        nrows = shape[0]
        mean_lat = y_first + (nrows / 2.0) * float(attrs["Y_STEP"])
        km_per_deg_lat = 111.32
        km_per_deg_lon = 111.32 * np.cos(np.deg2rad(mean_lat))
        dy_km = y_step * km_per_deg_lat
        dx_km = x_step * km_per_deg_lon
    else:
        # Projected map coordinates, commonly meters for UTM / EPSG products.
        dy_km = y_step / 1000.0
        dx_km = x_step / 1000.0

    return dy_km, dx_km, "km"

def plot_cross_section_km(data, start_yx, end_yx, attrs, n=500, ylabel="Value", title="Cross section", coherence=None):
    """
    Plot a cross section with distance in approximate kilometers.
    Works for projected-meter products and geographic lon/lat products using estimate_pixel_spacing_km().
    """
    dy_km, dx_km, units = estimate_pixel_spacing_km(attrs, data.shape)
    dist_pix, vals, ys, xs = sample_profile(data, start_yx, end_yx, n=n)

    # Convert cumulative pixel path to km using local dx/dy.
    dys = np.diff(ys) * dy_km
    dxs = np.diff(xs) * dx_km
    dist_km = np.concatenate([[0.0], np.cumsum(np.sqrt(dys**2 + dxs**2))])

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(dist_km, vals, "k-", linewidth=1.5)
    ax1.set_xlabel("Distance along profile, km")
    ax1.set_ylabel(ylabel)
    ax1.set_title(title)
    ax1.grid(True, alpha=0.3)

    if coherence is not None:
        _, coh_vals, _, _ = sample_profile(coherence, start_yx, end_yx, n=n)
        ax2 = ax1.twinx()
        ax2.plot(dist_km, coh_vals, color="0.6", linewidth=1)
        ax2.set_ylabel("Coherence / temporal coherence")
        ax2.set_ylim(0, 1)

    plt.show()
    return dist_km, vals

# Choose the map to profile:
# For Ridgecrest, use step_cm if defined; otherwise use velocity.
profile_map = step_cm if "step_cm" in globals() else vel_cm_yr
profile_label = "cm" if "step_cm" in globals() else "cm/year"

# EDIT THESE PROFILE ENDPOINTS.
profiles = {
    "Profile A": (),
    "Profile B": (),
    "Profile C": (),
}

for name, (start, end) in profiles.items():
    plot_profile_on_map(profile_map, start, end, title=f"{name}: location", label=profile_label)
    plot_cross_section_km(
        profile_map, start, end, vel_attrs,
        ylabel=f"LOS displacement/velocity ({profile_label})",
        title=f"{name}: cross section with distance in km",
        coherence=temporal_coh if temporal_coh is not None else None
    )


### Cross-section tasks

Make at least three cross sections, one across each of the three main displacement steps.

**QUESTION (5pts): Is this earthquake consistent with a single continuous fault or is it a complex rupture involving multiple faults, and if so, which section of the fault experienced the largest rupture?**

**ANSWER:**


## 8. Velocity gradient: highlighting creeping faults

In this section, I'm assuming you've already explored the mintpy plots for the San Francisco inversion. If not, look at those a bit to familiarize yourself with what you're looking at. This notebook will do some additional calculations on those calculated products.

A creeping fault can appear as a sharp spatial gradient in LOS velocity.  
The simplest diagnostic is the magnitude of the velocity gradient

This is not exactly geologic strain unless you carefully define direction, geometry, and displacement components. But it is a very useful image-processing diagnostic for finding narrow creeping fault deformation zones.


In [ ]:
DATA_DIR = Path("./SanFranSenDT42/mintpy").resolve()

# Main MintPy files
TS_FILE = DATA_DIR / "timeseries.h5"
VEL_FILE = DATA_DIR / "velocity.h5"
TCOH_FILE = DATA_DIR / "temporalCoherence.h5"
AVG_COH_FILE = DATA_DIR / "avgSpatialCoh.h5"

# Geometry can be in inputs/geometryGeo.h5 or geometryRadar.h5 depending on the stack
GEOM_CANDIDATES = [
    DATA_DIR / "inputs" / "geometryGeo.h5",
    DATA_DIR / "inputs" / "geometryRadar.h5",
    DATA_DIR / "geometryGeo.h5",
    DATA_DIR / "geometryRadar.h5",
]
GEOM_FILE = next((p for p in GEOM_CANDIDATES if p.exists()), None)

print("Working directory:", DATA_DIR)
print("timeseries.h5 exists:", TS_FILE.exists())
print("velocity.h5 exists:", VEL_FILE.exists())
print("geometry file:", GEOM_FILE)

#Load SF data
ts_cm, dates, ts_attrs = read_timeseries_cm(TS_FILE, reference_to_first=True)
vel_cm_yr, vel_attrs = read_velocity_cm_per_year(VEL_FILE)

temporal_coh, _ = read_optional_map(TCOH_FILE)
avg_coh, _ = read_optional_map(AVG_COH_FILE)

print("Time-series shape:", ts_cm.shape, "(date, y, x)")
print("Velocity shape:", vel_cm_yr.shape)
print("Dates:")
print_dates(dates)

print("\nReference date in file:", ts_attrs.get("REF_DATE"))
print("Reference pixel y/x:", ts_attrs.get("REF_Y"), ts_attrs.get("REF_X"))
print("Heading:", ts_attrs.get("HEADING"))


In [ ]:
def velocity_gradient(vel_cm_yr, attrs):
    """
    Calculate velocity gradient components and magnitude.
    Returns grad_mag, dv_dy, dv_dx, spacing_units.
    """
    dy, dx, units = estimate_pixel_spacing_km(attrs, vel_cm_yr.shape)
    print(f"Estimated pixel spacing: dy={dy:.4f} km, dx={dx:.4f} km")
    # np.gradient arguments are spacing along axis 0 and axis 1.
    dv_dy, dv_dx = np.gradient(vel_cm_yr, dy, dx)
    grad_mag = np.sqrt(dv_dx**2 + dv_dy**2)
    return grad_mag, dv_dy, dv_dx, units

grad_mag, dv_dy, dv_dx, spacing_units = velocity_gradient(vel_cm_yr, vel_attrs)

if spacing_units == "km":
    grad_label = "LOS velocity gradient, cm/year/km"
else:
    grad_label = "LOS velocity gradient, cm/year/pixel"

# Robust color limit for gradient magnitude
finite = grad_mag[np.isfinite(grad_mag)]
vmax_grad = np.nanpercentile(finite, 98) if finite.size else 1

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grad_mag, cmap="magma", vmin=0, vmax=vmax_grad)
ax.set_title("Magnitude of LOS velocity gradient")
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
plt.colorbar(im, ax=ax, label=grad_label)
plt.show()


In [ ]:
# Mask gradient by temporal coherence to suppress noisy areas.
COH_THRESHOLD = 0.70

if temporal_coh is not None:
    grad_masked = np.where(temporal_coh >= COH_THRESHOLD, grad_mag, np.nan)
    finite = grad_masked[np.isfinite(grad_masked)]
    vmax_grad_masked = np.nanpercentile(finite, 98) if finite.size else vmax_grad

    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(grad_masked, cmap="magma", vmin=0, vmax=vmax_grad_masked)
    ax.set_title(f"LOS velocity gradient masked by temporal coherence ≥ {COH_THRESHOLD}")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    plt.colorbar(im, ax=ax, label=grad_label)
    plt.show()
else:
    print("No temporal coherence file found; skipping coherence-masked gradient.")


**QUESTION (10pts): How many creeping faults can you identify. What are the features you are looking for?**

**ASNWER:**

**CODING TASK (20pts): In the cells above, we plotted displacement cross-sections across the faults. Here, I want you to plot a cross section across a creeping section of a fault, which are highlighted by high velocity gradients and appear as linear features in the velocity gradient map. Plot two versions of your chosen cross section, one in velocity, and one in velocity gradient.**

Note: A swath profile would be best here, but a bit more complicated. If you want to make a swath profile, it will reduce noise and paint a clearer picture of the fault. Swath profiles are very useful in noisy environments, and and excellent tool to add to your kit.

In [ ]:
###Your Code Here
# Single-line transect across a creeping fault section.
# Coordinates are image coordinates: (y, x), not (x, y).
# Choose a line approximately perpendicular to the fault trace.

PROFILE_NAME = "Creeping fault transect"

START_YX = ()   # <-- edit
END_YX   = ()   # <-- edit

N_SAMPLES = 800

USE_COH_MASK = True
COH_THRESHOLD = 0.70


# ------------------------------------------------------------
# Sample velocity and velocity-gradient along the same line
# ------------------------------------------------------------
dist_pix, vel_profile, ys, xs = sample_profile(
    vel_cm_yr,
    START_YX,
    END_YX,
    n=N_SAMPLES,
)

_, grad_profile, _, _ = sample_profile(
    grad_mag,
    START_YX,
    END_YX,
    n=N_SAMPLES,
)


# ------------------------------------------------------------
# Convert profile distance from pixels to km
# ------------------------------------------------------------
dy_km, dx_km, _ = estimate_pixel_spacing_km(vel_attrs, vel_cm_yr.shape)

dys_km = np.diff(ys) * dy_km
dxs_km = np.diff(xs) * dx_km

dist_km = np.concatenate([
    [0.0],
    np.cumsum(np.sqrt(dys_km**2 + dxs_km**2))
])


# ------------------------------------------------------------
# Optional coherence mask along the profile
# ------------------------------------------------------------
if USE_COH_MASK and temporal_coh is not None:
    _, coh_profile, _, _ = sample_profile(
        temporal_coh,
        START_YX,
        END_YX,
        n=N_SAMPLES,
    )

    good = coh_profile >= COH_THRESHOLD

    vel_profile_plot = np.where(good, vel_profile, np.nan)
    grad_profile_plot = np.where(good, grad_profile, np.nan)

else:
    coh_profile = None
    vel_profile_plot = vel_profile
    grad_profile_plot = grad_profile


# ------------------------------------------------------------
# Plot profile location on velocity map
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))

finite = vel_cm_yr[np.isfinite(vel_cm_yr)]
p = np.nanpercentile(np.abs(finite), 98) if finite.size else 1

im = ax.imshow(
    grad_masked,
    cmap="magma",
    vmin=0,
    vmax=vmax_grad_masked,
)

ax.plot(xs, ys, "r-", linewidth=2)
ax.plot(xs[0], ys[0], "ro", label="start")
ax.plot(xs[-1], ys[-1], "rs", label="end")

ax.set_title(f"{PROFILE_NAME}: line location")
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
ax.legend()

plt.colorbar(im, ax=ax, label="LOS velocity, cm/year")
plt.show()


# ------------------------------------------------------------
# Plot velocity and velocity-gradient profiles
# ------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    1,
    figsize=(10, 7),
    sharex=True,
    constrained_layout=True,
)

# Velocity profile
axes[0].plot(dist_km, vel_profile_plot, "k-", linewidth=1.8)
axes[0].axhline(0, color="0.5", linewidth=1)
axes[0].set_ylabel("LOS velocity, cm/year")
axes[0].set_title(f"{PROFILE_NAME}: LOS velocity")
axes[0].grid(True, alpha=0.3)

# Gradient profile
axes[1].plot(dist_km, grad_profile_plot, "k-", linewidth=1.8)
axes[1].set_xlabel("Distance along profile, km")
axes[1].set_ylabel(grad_label)
axes[1].set_title(f"{PROFILE_NAME}: LOS velocity gradient")
axes[1].grid(True, alpha=0.3)

if USE_COH_MASK and temporal_coh is not None:
    axes[0].text(
        0.02,
        0.95,
        f"Masked where temporal coherence < {COH_THRESHOLD}",
        transform=axes[0].transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7),
    )

plt.show()

## 11. Useful MintPy command-line tools

Run these from the MintPy result directory.

```bash
# Interactive map and point time-series viewer
tsview.py timeseries.h5

# View velocity and coherence products
view.py velocity.h5
view.py temporalCoherence.h5
view.py avgSpatialCoh.h5

# Plot the interferogram network
plot_network.py inputs/ifgramStack.h5

# Inspect file metadata
info.py timeseries.h5
info.py velocity.h5

# Re-reference time series to a different date
reference_date.py timeseries.h5 -r 20190610

# Re-reference to a different spatial point
reference_point.py timeseries.h5 --yx 500 700
```

Use the exact date and pixel coordinates appropriate for your dataset.
